# Personal Movie Recommender

This project explores different recommendation approaches to build a
personalized movie recommender based on my own movie ratings.

The project starts by benchmarking collaborative filtering on the MovieLens
dataset. Several approaches are then tested for personal recommendation,
including Bayesian Personalized Ranking, genre-based content filtering, and
item-item collaborative filtering.

The final recommender uses item-item collaborative filtering. Movie
similarities are learned from collective MovieLens behaviour and combined
with my own ratings to generate personalized recommendations.

## Project Structure

| Section | Model | Purpose |
|---|---|---|
| A–H | MovieLens BPR | Benchmark collaborative filtering |
| I1–I7 | Personalized BPR | Test direct personalization |
| J1–J3 | Genre-based | Test content-based personalization |
| K1–K11 | Item-item CF | Final recommender |

In [2]:
# ===== Step A: Data Prep  =====
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder

# Base folder where your notebook lives
BASE_DIR = r"C:\Users\RubenPoelhekke\Documents\internship_files\recommendation_project_ruben\notebooks"

# MovieLens dataset path
MOVIELENS_PATH = os.path.join(BASE_DIR, "ml-latest-small")

# Check files exist
if not os.path.exists(MOVIELENS_PATH):
    raise FileNotFoundError(f"MovieLens folder not found at {MOVIELENS_PATH}. Please check the path.")

print("Files in MovieLens folder:", os.listdir(MOVIELENS_PATH))

# Load MovieLens CSV files
ratings = pd.read_csv(os.path.join(MOVIELENS_PATH, "ratings.csv"))
movies = pd.read_csv(os.path.join(MOVIELENS_PATH, "movies.csv"))
links = pd.read_csv(os.path.join(MOVIELENS_PATH, "links.csv"))

print("MovieLens ratings loaded:", len(ratings))
print("Movies loaded:", len(movies))

# Load your personal ratings
MY_RATINGS_PATH = os.path.join(BASE_DIR, "my_ratings.csv")
if not os.path.exists(MY_RATINGS_PATH):
    raise FileNotFoundError(f"My ratings file not found at {MY_RATINGS_PATH}. Please check the path.")

my_ratings = pd.read_csv(MY_RATINGS_PATH)
my_user_id = ratings["userId"].max() + 1  # assign a new userId for yourself
my_ratings["userId"] = my_user_id

# Combine your ratings with original ratings
ratings_with_you = pd.concat([ratings, my_ratings], ignore_index=True)

# Encode userId and movieId as integers
user_enc = LabelEncoder()
movie_enc = LabelEncoder()
ratings_with_you['userId'] = ratings_with_you['userId'].astype(str)
ratings_with_you['movieId'] = ratings_with_you['movieId'].astype(str)
ratings_with_you['user_idx'] = user_enc.fit_transform(ratings_with_you['userId'])
ratings_with_you['movie_idx'] = movie_enc.fit_transform(ratings_with_you['movieId'])

num_users = ratings_with_you['user_idx'].nunique()
num_items = ratings_with_you['movie_idx'].nunique()
print("Number of users:", num_users, "Number of items:", num_items)

# Map back functions
movieid_to_itemidx = dict(zip(ratings_with_you['movieId'], ratings_with_you['movie_idx']))
itemidx_to_movieid = {v: k for k, v in movieid_to_itemidx.items()}

# Prepare genre matrix
movies['movieId'] = movies['movieId'].astype(str)
movies['genre_list'] = movies['genres'].fillna('').apply(lambda x: x.split('|') if x else [])
mlb = MultiLabelBinarizer()
mlb.fit(movies['genre_list'])
num_genres = len(mlb.classes_)
movieid_to_genre = dict(zip(movies['movieId'], mlb.transform(movies['genre_list'])))

# Oversample personal ratings

# Define oversample factor
oversample_factor = 30

# Separate base ratings (original dataset) from personal ratings
base_ratings = ratings.copy()
base_ratings['userId'] = base_ratings['userId'].astype(str)
base_ratings['movieId'] = base_ratings['movieId'].astype(str)
base_ratings['rating'] = base_ratings['rating'].astype(float)

my_ratings_str = my_ratings.copy()
my_ratings_str['userId'] = my_ratings_str['userId'].astype(str)
my_ratings_str['movieId'] = my_ratings_str['movieId'].astype(str)
my_ratings_str['rating'] = my_ratings_str['rating'].astype(float)

# Oversample your ratings
my_ratings_oversampled = pd.concat([my_ratings_str]*oversample_factor, ignore_index=True)

# Combine base ratings + oversampled personal ratings
train_ratings_df = pd.concat([base_ratings, my_ratings_oversampled], ignore_index=True).reset_index(drop=True)

# Re-encode to indices
train_ratings_df['user_idx'] = user_enc.transform(train_ratings_df['userId'])
train_ratings_df['movie_idx'] = movie_enc.transform(train_ratings_df['movieId'])

print("Training dataset prepared with oversampled personal ratings. Total rows:", len(train_ratings_df))
print("Step A completed: Data loaded and preprocessed.")


Files in MovieLens folder: ['links.csv', 'movies.csv', 'ratings.csv', 'requirements.txt', 'tags.csv']
MovieLens ratings loaded: 100836
Movies loaded: 9742
Number of users: 611 Number of items: 9724
Training dataset prepared with oversampled personal ratings. Total rows: 106806
Step A completed: Data loaded and preprocessed.


In [ ]:
# ===== Step B: Create train/test split =====
from collections import defaultdict

# Use only MovieLens ratings for the benchmark
eval_ratings = ratings.copy()

# Convert timestamp to datetime
eval_ratings["timestamp"] = pd.to_datetime(
    eval_ratings["timestamp"], unit="s"
)

# Only consider ratings >= 4 as positive interactions
eval_ratings = eval_ratings[eval_ratings["rating"] >= 4].copy()

# Convert IDs to the same format as the encoders
eval_ratings["userId"] = eval_ratings["userId"].astype(str)
eval_ratings["movieId"] = eval_ratings["movieId"].astype(str)

eval_ratings["user_idx"] = user_enc.transform(eval_ratings["userId"])
eval_ratings["movie_idx"] = movie_enc.transform(eval_ratings["movieId"])

# Train/test dictionaries
user_pos_train = defaultdict(set)
user_pos_test = defaultdict(set)

# Temporal split: last 20% of each user's positive ratings = test
for user_idx, user_ratings in eval_ratings.groupby("user_idx"):

    user_ratings = user_ratings.sort_values("timestamp")

    n = len(user_ratings)

    if n < 2:
        user_pos_train[user_idx] = set(user_ratings["movie_idx"])
        continue

    split_idx = max(1, int(n * 0.8))

    train_ratings = user_ratings.iloc[:split_idx]
    test_ratings = user_ratings.iloc[split_idx:]

    user_pos_train[user_idx] = set(train_ratings["movie_idx"])
    user_pos_test[user_idx] = set(test_ratings["movie_idx"])

print("Users:", len(user_pos_train))
print("Users with test data:", sum(len(x) > 0 for x in user_pos_test.values()))
print("Training interactions:", sum(len(x) for x in user_pos_train.values()))
print("Test interactions:", sum(len(x) for x in user_pos_test.values()))

In [ ]:
# ===== Step C: Create BPR Training Pairs =====
import numpy as np

# Positive interactions available for training only
user_pos = user_pos_train

# All movie indices
all_item_idxs = np.arange(num_items)

# Flatten user-positive pairs
user_pos_pairs = []

for u, pos_items in user_pos.items():
    for i in pos_items:
        user_pos_pairs.append((u, i))

user_pos_pairs = np.array(user_pos_pairs, dtype=np.int64)

print("Training pairs:", len(user_pos_pairs))
print("Users:", len(user_pos))


def sample_triplets(num_samples):
    """
    Sample (user, positive item, negative item) triplets
    using training interactions only.
    """

    # Randomly select training user-positive pairs
    indices = np.random.randint(
        0,
        len(user_pos_pairs),
        size=num_samples
    )

    selected_pairs = user_pos_pairs[indices]

    users = selected_pairs[:, 0]
    positive_items = selected_pairs[:, 1]

    # Random negative items
    negative_items = np.random.randint(
        0,
        num_items,
        size=num_samples
    )

    # Make sure negative item was not in the user's training positives
    for idx in range(num_samples):
        while negative_items[idx] in user_pos[users[idx]]:
            negative_items[idx] = np.random.randint(0, num_items)

    return users, positive_items, negative_items

Training pairs: 38615
Users: 609


In [ ]:
# ===== Step D: Define BPR Model =====
import torch
import torch.nn as nn
import torch.optim as optim

class BPRModel(nn.Module):
    def __init__(self, num_users, num_items, emb_dim=64):
        super().__init__()

        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.item_emb = nn.Embedding(num_items, emb_dim)

        nn.init.normal_(self.user_emb.weight, std=0.02)
        nn.init.normal_(self.item_emb.weight, std=0.02)

    def forward(self, u, pos_i, neg_i):

        u_emb = self.user_emb(u)
        p_emb = self.item_emb(pos_i)
        n_emb = self.item_emb(neg_i)

        pos_score = (u_emb * p_emb).sum(dim=1)
        neg_score = (u_emb * n_emb).sum(dim=1)

        return pos_score, neg_score


# Create a completely fresh model
model = BPRModel(
    num_users=num_users,
    num_items=num_items,
    emb_dim=64
)

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Model created.")
print("Users:", num_users)
print("Items:", num_items)

Model created.
Users: 611
Items: 9724


In [ ]:
# ===== Step E: Train BPR =====

import torch
import numpy as np

# -------- Training pairs from TRAIN data only --------

all_users = []
all_pos = []

for u, pos_items in user_pos_train.items():
    all_users.extend([u] * len(pos_items))
    all_pos.extend(pos_items)

all_users = np.array(all_users, dtype=np.int64)
all_pos = np.array(all_pos, dtype=np.int64)

print("Training pairs:", len(all_users))


# -------- Negative sampler --------

def sample_triplets_no_loop(num_samples):
    """
    Sample (user, positive item, negative item) triplets
    using training interactions only.
    """

    # Sample user-positive pairs
    idx = np.random.randint(
        0,
        len(all_users),
        size=num_samples
    )

    u_array = all_users[idx]
    p_array = all_pos[idx]

    # Sample random negative items
    neg_array = np.random.randint(
        0,
        num_items,
        size=num_samples
    )

    # Make sure negatives are not training positives
    mask = np.array([
        neg in user_pos_train[u]
        for u, neg in zip(u_array, neg_array)
    ])

    while mask.any():
        neg_array[mask] = np.random.randint(
            0,
            num_items,
            size=mask.sum()
        )

        mask = np.array([
            neg in user_pos_train[u]
            for u, neg in zip(u_array, neg_array)
        ])

    return (
        u_array.astype(np.int64),
        p_array.astype(np.int64),
        neg_array.astype(np.int64)
    )


# -------- Training parameters --------

EPOCHS = 15
BATCH_SIZE = 2048
STEPS_PER_EPOCH = 800

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)
model.train()

print("Device:", device)


# -------- Training loop --------

for epoch in range(1, EPOCHS + 1):

    epoch_loss = 0.0

    for step in range(STEPS_PER_EPOCH):

        u, p, n = sample_triplets_no_loop(BATCH_SIZE)

        u = torch.tensor(
            u,
            dtype=torch.long,
            device=device
        )

        p = torch.tensor(
            p,
            dtype=torch.long,
            device=device
        )

        n = torch.tensor(
            n,
            dtype=torch.long,
            device=device
        )

        optimizer.zero_grad()

        pos_score, neg_score = model(u, p, n)

        # BPR loss
        loss = -torch.log(
            torch.sigmoid(pos_score - neg_score) + 1e-8
        ).mean()

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        if step % 200 == 0:
            print(
                f"Epoch {epoch} "
                f"Step {step}/{STEPS_PER_EPOCH} "
                f"Loss: {loss.item():.4f}"
            )

    print(
        f"==== Epoch {epoch} Completed. "
        f"Mean loss: {epoch_loss / STEPS_PER_EPOCH:.4f} ===="
    )

print("Training completed.")

Training pairs: 38615
Device: cpu
Epoch 1 Step 0/800 Loss: 0.6931
Epoch 1 Step 200/800 Loss: 0.3003
Epoch 1 Step 400/800 Loss: 0.1681
Epoch 1 Step 600/800 Loss: 0.1214
==== Epoch 1 Completed. Mean loss: 0.2620 ====
Epoch 2 Step 0/800 Loss: 0.1002
Epoch 2 Step 200/800 Loss: 0.0860
Epoch 2 Step 400/800 Loss: 0.0704
Epoch 2 Step 600/800 Loss: 0.0554
==== Epoch 2 Completed. Mean loss: 0.0748 ====
Epoch 3 Step 0/800 Loss: 0.0559
Epoch 3 Step 200/800 Loss: 0.0536
Epoch 3 Step 400/800 Loss: 0.0406
Epoch 3 Step 600/800 Loss: 0.0388
==== Epoch 3 Completed. Mean loss: 0.0428 ====
Epoch 4 Step 0/800 Loss: 0.0301
Epoch 4 Step 200/800 Loss: 0.0302
Epoch 4 Step 400/800 Loss: 0.0273
Epoch 4 Step 600/800 Loss: 0.0250
==== Epoch 4 Completed. Mean loss: 0.0278 ====
Epoch 5 Step 0/800 Loss: 0.0253
Epoch 5 Step 200/800 Loss: 0.0198
Epoch 5 Step 400/800 Loss: 0.0167
Epoch 5 Step 600/800 Loss: 0.0193
==== Epoch 5 Completed. Mean loss: 0.0192 ====
Epoch 6 Step 0/800 Loss: 0.0146
Epoch 6 Step 200/800 Loss: 0.

In [ ]:
# ===== Step F: Generate Recommendations =====

model.eval()

# Your MovieLens user for evaluation
eval_user_idx = user_enc.transform(["1"])[0]

with torch.no_grad():

    # User embedding
    user_embedding = model.user_emb(
        torch.tensor([eval_user_idx], dtype=torch.long, device=device)
    )

    # All item embeddings
    item_embeddings = model.item_emb.weight

    # Score every movie
    scores = torch.matmul(
        user_embedding,
        item_embeddings.T
    ).squeeze(0)

    # Movies already seen in TRAIN
    seen_items = user_pos_train[eval_user_idx]

    # Don't recommend training movies
    scores[list(seen_items)] = -float("inf")

    # Get top 150 recommendations
    top_k = 150

    top_scores, top_indices = torch.topk(
        scores,
        k=top_k
    )

recommended_item_idxs = top_indices.cpu().numpy()
recommended_scores = top_scores.cpu().numpy()

# Convert item indices back to MovieLens movie IDs
recommended_movie_ids = [
    itemidx_to_movieid[idx]
    for idx in recommended_item_idxs
]

# Create recommendation dataframe
recommendations = pd.DataFrame({
    "movieId": recommended_movie_ids,
    "score": recommended_scores
})

recommendations = recommendations.merge(
    movies[["movieId", "title", "genres"]],
    on="movieId",
    how="left"
)

recommendations.head(20)

,movieId,score,title,genres
0,2,9.455785,Jumanji (1995),Adventure|Children|Fantasy
1,1580,9.430957,Men in Black (a.k.a. MIB) (1997),Action|Comedy|Sci-Fi
2,1036,9.134309,Die Hard (1988),Action|Crime|Thriller
3,296,9.016010,Pulp Fiction (1994),Comedy|Crime|Drama|Thriller
4,1394,8.857565,Raising Arizona (1987),Comedy
5,1240,8.394691,"Terminator, The (1984)",Action|Sci-Fi|Thriller
6,368,8.353586,Maverick (1994),Adventure|Comedy|Western
7,1265,8.114050,Groundhog Day (1993),Comedy|Fantasy|Romance
8,47,8.104559,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
9,780,8.054214,Independence Day (a.k.a. ID4) (1996),Action|Adventure|Sci-Fi|Thriller


In [15]:
# ===== Step G: Evaluation Metrics =====

import numpy as np

K = 10


def apk(actual, predicted, k=10):
    """
    Average Precision at K for one user.
    """
    if len(actual) == 0:
        return 0.0

    predicted = predicted[:k]

    score = 0.0
    hits = 0

    for i, p in enumerate(predicted):
        if p in actual:
            hits += 1
            score += hits / (i + 1)

    return score / min(len(actual), k)


def ndcg_at_k(actual, predicted, k=10):
    """
    NDCG@K with binary relevance.
    """
    predicted = predicted[:k]

    dcg = 0.0

    for i, p in enumerate(predicted):
        if p in actual:
            dcg += 1 / np.log2(i + 2)

    ideal_hits = min(len(actual), k)

    idcg = sum(
        1 / np.log2(i + 2)
        for i in range(ideal_hits)
    )

    if idcg == 0:
        return 0.0

    return dcg / idcg


# Users that actually have held-out test interactions
evaluation_users = [
    u for u in user_pos_test
    if len(user_pos_test[u]) > 0
]

print("Users evaluated:", len(evaluation_users))


# Store metrics
hit_rates = []
average_precisions = []
ndcgs = []


model.eval()

with torch.no_grad():

    # All item embeddings
    item_embeddings = model.item_emb.weight

    for user_idx in evaluation_users:

        # User embedding
        user_tensor = torch.tensor(
            [user_idx],
            dtype=torch.long,
            device=device
        )

        user_embedding = model.user_emb(user_tensor)

        # Score every movie
        scores = torch.matmul(
            user_embedding,
            item_embeddings.T
        ).squeeze(0).clone()

        # Remove movies already seen in TRAIN
        train_items = user_pos_train[user_idx]

        if len(train_items) > 0:
            scores[list(train_items)] = -float("inf")

        # Top K recommendations
        top_indices = torch.topk(
            scores,
            k=K
        ).indices.cpu().numpy()

        # Movies the user actually liked in TEST
        actual = user_pos_test[user_idx]

        # HR@K
        hit = int(
            any(item in actual for item in top_indices)
        )

        hit_rates.append(hit)

        # MAP@K
        average_precisions.append(
            apk(actual, top_indices, K)
        )

        # NDCG@K
        ndcgs.append(
            ndcg_at_k(actual, top_indices, K)
        )


# Final metrics
hr = np.mean(hit_rates)
map_k = np.mean(average_precisions)
ndcg = np.mean(ndcgs)


print("\n===== BPR Evaluation =====")
print(f"HR@{K}:   {hr:.4f}")
print(f"MAP@{K}:  {map_k:.4f}")
print(f"NDCG@{K}: {ndcg:.4f}")

Users evaluated: 608

===== BPR Evaluation =====
HR@10:   0.4326
MAP@10:  0.0374
NDCG@10: 0.0856


In [16]:
# ===== Step H: Popularity Baseline =====

from collections import Counter
import numpy as np

K = 10

# Count how many users rated each movie positively in the TRAIN set
item_popularity = Counter()

for user_idx, items in user_pos_train.items():
    for item_idx in items:
        item_popularity[item_idx] += 1

# Rank movies by popularity
popular_items = [
    item_idx
    for item_idx, count in item_popularity.most_common()
]

print("Unique popular items:", len(popular_items))


# Store baseline metrics
pop_hit_rates = []
pop_average_precisions = []
pop_ndcgs = []


# Evaluate the popularity recommender
for user_idx in evaluation_users:

    # Movies already seen by this user during training
    train_items = user_pos_train[user_idx]

    # Recommend most popular movies the user has NOT seen
    recommendations = [
        item_idx
        for item_idx in popular_items
        if item_idx not in train_items
    ][:K]

    # Actual held-out positive interactions
    actual = user_pos_test[user_idx]

    # HR@K
    hit = int(
        any(item in actual for item in recommendations)
    )

    pop_hit_rates.append(hit)

    # MAP@K
    pop_average_precisions.append(
        apk(actual, recommendations, K)
    )

    # NDCG@K
    pop_ndcgs.append(
        ndcg_at_k(actual, recommendations, K)
    )


# Final baseline metrics
pop_hr = np.mean(pop_hit_rates)
pop_map = np.mean(pop_average_precisions)
pop_ndcg = np.mean(pop_ndcgs)


print("\n===== Popularity Baseline =====")
print(f"HR@{K}:   {pop_hr:.4f}")
print(f"MAP@{K}:  {pop_map:.4f}")
print(f"NDCG@{K}: {pop_ndcg:.4f}")

Unique popular items: 5290

===== Popularity Baseline =====
HR@10:   0.3141
MAP@10:  0.0325
NDCG@10: 0.0677


In [ ]:
# ===== Personal Model: Step I1 =====

print("Your ratings:", len(my_ratings))
print(
    "Your positive ratings:",
    (my_ratings["rating"] >= 4).sum()
)
print(
    "Your lower ratings:",
    (my_ratings["rating"] < 4).sum()
)

Your ratings: 199
Your positive ratings: 83
Your lower ratings: 116


In [ ]:
# ===== Personal Model: Step I2 =====

from sklearn.model_selection import train_test_split

# Make a copy of your ratings
personal_ratings = my_ratings.copy()

# Keep only positive ratings for BPR
personal_positive = personal_ratings[
    personal_ratings["rating"] >= 4
].copy()

# Convert movie IDs to the same format used by the model
personal_positive["movieId"] = (
    personal_positive["movieId"]
    .astype(str)
)

# Split your positive ratings
personal_train, personal_test = train_test_split(
    personal_positive,
    test_size=0.20,
    random_state=42
)

# Convert to item indices
personal_train["movie_idx"] = movie_enc.transform(
    personal_train["movieId"]
)

personal_test["movie_idx"] = movie_enc.transform(
    personal_test["movieId"]
)

# Sets for easy lookup
personal_train_pos = set(
    personal_train["movie_idx"]
)

personal_test_pos = set(
    personal_test["movie_idx"]
)

# All movies you have already rated
personal_rated = set(
    movie_enc.transform(
        personal_ratings["movieId"].astype(str)
    )
)

print("Your positive ratings:", len(personal_positive))
print("Your positive training ratings:", len(personal_train_pos))
print("Your positive test ratings:", len(personal_test_pos))
print("Your total rated movies:", len(personal_rated))

Your positive ratings: 83
Your positive training ratings: 66
Your positive test ratings: 17
Your total rated movies: 199


In [ ]:
# ===== Personal Model: Step I3 =====

# Start with MovieLens training interactions
personal_user_pos = {
    u: set(items)
    for u, items in user_pos_train.items()
}

# Add your positive training ratings
personal_user_pos[my_idx] = set(personal_train_pos)

# Create flattened user-positive pairs
all_users = []
all_pos = []

for u, pos_items in personal_user_pos.items():
    all_users.extend([u] * len(pos_items))
    all_pos.extend(pos_items)

all_users = np.array(all_users, dtype=np.int64)
all_pos = np.array(all_pos, dtype=np.int64)

print("Total training pairs:", len(all_users))
print("MovieLens training pairs:", sum(
    len(items) for items in user_pos_train.values()
))
print("Your training pairs:", len(personal_train_pos))
print("Your user index:", my_idx)

def sample_triplets_personal(num_samples):
    """
    Sample BPR triplets using MovieLens training data
    plus your personal positive training ratings.
    """

    # Sample user-positive pairs
    idx = np.random.randint(
        0,
        len(all_users),
        size=num_samples
    )

    u_array = all_users[idx]
    p_array = all_pos[idx]

    # Random negative items
    neg_array = np.random.randint(
        0,
        num_items,
        size=num_samples
    )

    # Make sure negatives are not positive training items
    mask = np.array([
        neg in personal_user_pos[u]
        for u, neg in zip(u_array, neg_array)
    ])

    while mask.any():
        neg_array[mask] = np.random.randint(
            0,
            num_items,
            size=mask.sum()
        )

        mask = np.array([
            neg in personal_user_pos[u]
            for u, neg in zip(u_array, neg_array)
        ])

    return (
        u_array.astype(np.int64),
        p_array.astype(np.int64),
        neg_array.astype(np.int64)
    )

Total training pairs: 38681
MovieLens training pairs: 38615
Your training pairs: 66
Your user index: 569


In [ ]:
# ===== Personal Model: Step I4 =====

PERSONAL_WEIGHT = 10

# MovieLens training pairs
movielens_users = []
movielens_pos = []

for u, pos_items in user_pos_train.items():
    for item in pos_items:
        movielens_users.append(u)
        movielens_pos.append(item)

movielens_users = np.array(
    movielens_users,
    dtype=np.int64
)

movielens_pos = np.array(
    movielens_pos,
    dtype=np.int64
)

# Your positive training pairs
personal_users = np.full(
    len(personal_train_pos),
    my_idx,
    dtype=np.int64
)

personal_pos = np.array(
    list(personal_train_pos),
    dtype=np.int64
)


def sample_triplets_personal(num_samples):
    """
    Sample BPR triplets with extra weight on
    the personal user's positive interactions.
    """

    # Decide which samples come from your ratings
    personal_mask = (
        np.random.rand(num_samples)
        < (
            PERSONAL_WEIGHT * len(personal_pos)
            / (
                len(movielens_pos)
                + PERSONAL_WEIGHT * len(personal_pos)
            )
        )
    )

    num_personal = personal_mask.sum()
    num_movielens = num_samples - num_personal

    # Sample MovieLens pairs
    ml_indices = np.random.randint(
        0,
        len(movielens_pos),
        size=num_movielens
    )

    ml_users = movielens_users[ml_indices]
    ml_pos = movielens_pos[ml_indices]

    # Sample personal pairs
    personal_indices = np.random.randint(
        0,
        len(personal_pos),
        size=num_personal
    )

    p_users = personal_users[personal_indices]
    p_pos = personal_pos[personal_indices]

    # Combine
    u_array = np.concatenate([
        ml_users,
        p_users
    ])

    p_array = np.concatenate([
        ml_pos,
        p_pos
    ])

    # Shuffle the batch
    shuffle_idx = np.random.permutation(num_samples)

    u_array = u_array[shuffle_idx]
    p_array = p_array[shuffle_idx]

    # Sample negatives
    neg_array = np.random.randint(
        0,
        num_items,
        size=num_samples
    )

    # Make sure negatives aren't positive for that user
    mask = np.array([
        neg in personal_user_pos[u]
        for u, neg in zip(u_array, neg_array)
    ])

    while mask.any():

        neg_array[mask] = np.random.randint(
            0,
            num_items,
            size=mask.sum()
        )

        mask = np.array([
            neg in personal_user_pos[u]
            for u, neg in zip(u_array, neg_array)
        ])

    return (
        u_array.astype(np.int64),
        p_array.astype(np.int64),
        neg_array.astype(np.int64)
    )

# Check how often personal data will be sampled

personal_probability = (
    PERSONAL_WEIGHT * len(personal_pos)
    / (
        len(movielens_pos)
        + PERSONAL_WEIGHT * len(personal_pos)
    )
)

print(
    f"Personal sampling probability: "
    f"{personal_probability:.2%}"
)

In [ ]:
# ===== Personal Model: Step I5 =====

personal_model = BPRModel(
    num_users=num_users,
    num_items=num_items,
    emb_dim=64
)

personal_optimizer = optim.Adam(
    personal_model.parameters(),
    lr=0.001
)

personal_model = personal_model.to(device)

print("Personalized model created.")

Personalized model created.


In [ ]:
# ===== Personal Model Evaluation: Step I6 =====

personal_model.eval()

with torch.no_grad():

    user_tensor = torch.tensor(
        [my_idx],
        dtype=torch.long,
        device=device
    )

    # Score every movie for you
    user_embedding = personal_model.user_emb(user_tensor)
    item_embeddings = personal_model.item_emb.weight

    scores = (
        user_embedding @ item_embeddings.T
    ).squeeze(0).cpu().numpy()

# Movies that should NOT be candidates during evaluation:
# your positive training movies + your known lower-rated movies
personal_negative = set(
    personal_ratings.loc[
        personal_ratings["rating"] < 4,
        "movieId"
    ].astype(str)
)

personal_negative_idx = set(
    movie_enc.transform(list(personal_negative))
)

excluded_items = personal_train_pos | personal_negative_idx

# Keep your 17 hidden positive ratings as possible recommendations
candidate_items = [
    i for i in range(num_items)
    if i not in excluded_items
]

# Rank candidate movies
ranked_items = sorted(
    candidate_items,
    key=lambda x: scores[x],
    reverse=True
)

top_k = ranked_items[:10]

# Check how many hidden positives appear in top 10
hits = [
    item for item in top_k
    if item in personal_test_pos
]

hr10 = 1.0 if len(hits) > 0 else 0.0

print("Hidden positive ratings:", len(personal_test_pos))
print("Hits in top 10:", len(hits))
print(f"HR@10: {hr10:.4f}")

# Display the top 10 recommendations
top10_movie_ids = [
    itemidx_to_movieid[i]
    for i in top_k
]

personal_top10 = movies[
    movies["movieId"].isin(top10_movie_ids)
][
    ["movieId", "title", "genres"]
].copy()

personal_top10["score"] = personal_top10["movieId"].map(
    dict(
        zip(
            top10_movie_ids,
            [scores[i] for i in top_k]
        )
    )
)

personal_top10 = personal_top10.sort_values(
    "score",
    ascending=False
)

display(personal_top10)

Hidden positive ratings: 17
Hits in top 10: 0
HR@10: 0.0000


,movieId,title,genres,score
3190,4299,"Knight's Tale, A (2001)",Action|Comedy|Romance,0.012117
2136,2843,"Black Cat, White Cat (Crna macka, beli macor) ...",Comedy|Romance,0.010658
59,66,Lawnmower Man 2: Beyond Cyberspace (1996),Action|Sci-Fi|Thriller,0.010389
4813,7171,Aileen: Life and Death of a Serial Killer (2003),Documentary,0.010356
3496,4777,"American Astronaut, The (2001)",Comedy|Musical|Sci-Fi,0.010325
9107,144620,Goosebumps (2015),Adventure|Comedy|Horror,0.010138
9660,181315,Phantom Thread (2017),Drama|Romance,0.009982
7641,88069,Delhi Belly (2011),Comedy|Crime,0.009978
1674,2253,Toys (1992),Comedy|Fantasy,0.009932
2459,3272,Bad Lieutenant (1992),Crime|Drama,0.009918


In [ ]:
# ===== Personal Model Evaluation Metrics: Step I7 =====

def apk(actual, predicted, k=10):
    if len(actual) == 0:
        return 0.0

    predicted = predicted[:k]

    score = 0.0
    hits = 0

    for i, p in enumerate(predicted):
        if p in actual:
            hits += 1
            score += hits / (i + 1)

    return score / min(len(actual), k)


def ndcg_at_k(actual, predicted, k=10):
    predicted = predicted[:k]

    dcg = 0.0

    for i, p in enumerate(predicted):
        if p in actual:
            dcg += 1 / np.log2(i + 2)

    ideal_hits = min(len(actual), k)

    idcg = sum(
        1 / np.log2(i + 2)
        for i in range(ideal_hits)
    )

    if idcg == 0:
        return 0.0

    return dcg / idcg


personal_map10 = apk(
    personal_test_pos,
    top_k,
    k=10
)

personal_ndcg10 = ndcg_at_k(
    personal_test_pos,
    top_k,
    k=10
)

print(f"Personal HR@10:   {hr10:.4f}")
print(f"Personal MAP@10:  {personal_map10:.4f}")
print(f"Personal NDCG@10: {personal_ndcg10:.4f}")

Personal HR@10:   0.0000
Personal MAP@10:  0.0000
Personal NDCG@10: 0.0000


In [ ]:
# Step J1: ===== Personal Genre Profile =====

# Create genre matrix for all movies
genre_matrix = pd.DataFrame(
    mlb.transform(movies["genre_list"]),
    columns=mlb.classes_,
    index=movies["movieId"].astype(str)
)

# Add ratings to the genre matrix
personal_ratings_genre = personal_ratings.copy()
personal_ratings_genre["movieId"] = (
    personal_ratings_genre["movieId"].astype(str)
)

personal_ratings_genre = personal_ratings_genre.merge(
    movies[["movieId", "genres"]],
    on="movieId",
    how="left"
)

# Calculate average rating for each genre
genre_scores = {}

for genre in mlb.classes_:

    relevant_movies = personal_ratings_genre[
        personal_ratings_genre["genres"].str.contains(
            genre,
            regex=False,
            na=False
        )
    ]

    if len(relevant_movies) > 0:
        genre_scores[genre] = relevant_movies["rating"].mean()
    else:
        genre_scores[genre] = np.nan

genre_profile = (
    pd.Series(genre_scores)
    .sort_values(ascending=False)
)

print("Your genre preferences:")
display(
    genre_profile.to_frame("average_rating")
)

Your genre preferences:


,average_rating
Mystery,4.272727
Crime,4.269231
War,4.000000
(no genres listed),4.000000
Drama,3.990385
Thriller,3.750000
IMAX,3.637931
Romance,3.588235
Action,3.543210
Adventure,3.473333


In [ ]:
# Step J2: ===== Content-Based Movie Scoring =====

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# ===== Improved Content-Based Recommender =====

# Create a clean movie table
movie_features = movies[
    ["movieId", "title", "genres"]
].copy()

movie_features["movieId"] = (
    movie_features["movieId"].astype(str)
)

# Convert pipe-separated genres into space-separated tokens
movie_features["genre_text"] = (
    movie_features["genres"]
    .fillna("")
    .str.replace("|", " ", regex=False)
)

# TF-IDF representation of movie genres
tfidf = TfidfVectorizer(
    token_pattern=r"(?u)\b[\w-]+\b"
)

movie_tfidf = tfidf.fit_transform(
    movie_features["genre_text"]
)

# Map movie IDs to matrix rows
movie_id_to_row = dict(
    zip(
        movie_features["movieId"],
        range(len(movie_features))
    )
)

# Get your ratings for movies that exist in the feature matrix
rated = personal_ratings.copy()

rated["movieId"] = (
    rated["movieId"].astype(str)
)

rated = rated[
    rated["movieId"].isin(movie_id_to_row)
].copy()

# Build your personal taste vector
rated_rows = [
    movie_id_to_row[movie_id]
    for movie_id in rated["movieId"]
]

rated_matrix = movie_tfidf[rated_rows]

# Center ratings around your average
rating_weights = (
    rated["rating"].values
    - rated["rating"].mean()
)

# Weight movie features by your rating
weighted_profile = (
    rated_matrix.multiply(
        rating_weights[:, None]
    ).sum(axis=0)
)

# Convert to array
weighted_profile = np.asarray(
    weighted_profile
).reshape(1, -1)

# Calculate similarity between your profile
# and every movie
similarities = cosine_similarity(
    weighted_profile,
    movie_tfidf
).flatten()

# Create score series
content_scores = pd.Series(
    similarities,
    index=movie_features["movieId"]
)

# Exclude every movie you have already rated
rated_movie_ids = set(
    personal_ratings["movieId"].astype(str)
)

content_candidates = content_scores[
    ~content_scores.index.isin(rated_movie_ids)
]

# Top 20 recommendations
top_content = (
    content_candidates
    .sort_values(ascending=False)
    .head(20)
)

content_recommendations = movie_features[
    movie_features["movieId"].isin(top_content.index)
][
    ["movieId", "title", "genres"]
].copy()

content_recommendations["score"] = (
    content_recommendations["movieId"]
    .map(top_content)
)

content_recommendations = (
    content_recommendations
    .sort_values("score", ascending=False)
)

display(content_recommendations)

,movieId,title,genres,score
397,456,Fresh (1994),Crime|Drama|Thriller,0.644957
576,707,Mulholland Falls (1996),Crime|Drama|Thriller,0.644957
2306,3056,Oxygen (1999),Crime|Drama|Thriller,0.644957
2389,3169,The Falcon and the Snowman (1985),Crime|Drama|Thriller,0.644957
4095,5867,Thief (1981),Crime|Drama|Thriller,0.644957
4112,5893,"Last Seduction, The (1994)",Crime|Drama|Thriller,0.644957
4143,5959,Narc (2002),Crime|Drama|Thriller,0.644957
4172,6009,"City of Lost Souls, The (Hyôryuu-gai) (2000)",Crime|Drama|Thriller,0.644957
4206,6086,"I, the Jury (1982)",Crime|Drama|Thriller,0.644957
4336,6335,"Dancer Upstairs, The (2002)",Crime|Drama|Thriller,0.644957


In [ ]:
# Step J3: ===== Evaluate Content-Based Recommender =====

# Use the same train/test split as the personal BPR model

# Exclude movies from the personal training set
# and lower-rated movies from the candidate pool
personal_train_movie_ids = set(
    personal_train["movieId"].astype(str)
)

personal_low_rated_ids = set(
    personal_ratings.loc[
        personal_ratings["rating"] < 4,
        "movieId"
    ].astype(str)
)

excluded_for_eval = (
    personal_train_movie_ids
    | personal_low_rated_ids
)

content_eval_candidates = content_scores[
    ~content_scores.index.isin(excluded_for_eval)
]

content_ranked = (
    content_eval_candidates
    .sort_values(ascending=False)
)

content_top10 = list(
    content_ranked.head(10).index
)

# Metrics
content_hr10 = (
    1.0
    if any(
        movie_enc.transform([movie_id])[0]
        in personal_test_pos
        for movie_id in content_top10
        if movie_id in movieid_to_itemidx
    )
    else 0.0
)

content_top10_item_idx = [
    movieid_to_itemidx[movie_id]
    for movie_id in content_top10
    if movie_id in movieid_to_itemidx
]

content_map10 = apk(
    personal_test_pos,
    content_top10_item_idx,
    k=10
)

content_ndcg10 = ndcg_at_k(
    personal_test_pos,
    content_top10_item_idx,
    k=10
)

print(f"Content HR@10:   {content_hr10:.4f}")
print(f"Content MAP@10:  {content_map10:.4f}")
print(f"Content NDCG@10: {content_ndcg10:.4f}")

Content HR@10:   0.0000
Content MAP@10:  0.0000
Content NDCG@10: 0.0000


In [ ]:
# Step K1: Build the MovieLens item-item similarity model

from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

# Build MovieLens user-item matrix from positive training interactions
ml_train_rows = []
ml_train_cols = []

for u, items in user_pos_train.items():
    for item in items:
        ml_train_rows.append(u)
        ml_train_cols.append(item)

ml_user_item = csr_matrix(
    (
        np.ones(len(ml_train_rows)),
        (ml_train_rows, ml_train_cols)
    ),
    shape=(num_users, num_items)
)

# Transpose so each row represents one movie
ml_item_user = ml_user_item.T.tocsr()

print("User-item matrix shape:", ml_user_item.shape)
print("Item-user matrix shape:", ml_item_user.shape)
print("Non-zero interactions:", ml_item_user.nnz)

# Find the most similar movies for every movie
N_NEIGHBORS = 50

item_knn = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=N_NEIGHBORS + 1
)

item_knn.fit(ml_item_user)

distances, indices = item_knn.kneighbors(ml_item_user)

item_similarities = 1 - distances

print("Similarity matrix shape:", item_similarities.shape)

User-item matrix shape: (611, 9724)
Item-user matrix shape: (9724, 611)
Non-zero interactions: 38615
Similarity matrix shape: (9724, 51)


In [ ]:
# Step K2: Generate recommendations from your personal ratings

# Create mapping from item index to row in the KNN results
item_idx_to_knn_row = {
    item_idx: row_idx
    for row_idx, item_idx in enumerate(range(num_items))
}

# Personal training ratings with item indices
personal_profile = personal_train.copy()

personal_profile["movie_idx"] = movie_enc.transform(
    personal_profile["movieId"].astype(str)
)

# Score candidate movies based on similarity to movies you liked
cf_scores = defaultdict(float)

for _, row in personal_profile.iterrows():
    source_item = row["movie_idx"]
    
    # Give higher-rated movies more influence
    rating_weight = row["rating"] - 3
    
    knn_row = item_idx_to_knn_row[source_item]
    
    for similarity, similar_item in zip(
        item_similarities[knn_row][1:],
        indices[knn_row][1:]
    ):
        cf_scores[similar_item] += (
            similarity * rating_weight
        )

# Convert to Series
cf_scores = pd.Series(cf_scores)

# Exclude movies already rated positively in training
cf_scores = cf_scores[
    ~cf_scores.index.isin(personal_train_pos)
]

# Also exclude movies you rated below 4
cf_scores = cf_scores[
    ~cf_scores.index.isin(
        personal_ratings.loc[
            personal_ratings["rating"] < 4,
            "movieId"
        ].astype(str).map(movieid_to_itemidx)
    )
]

# Show top recommendations
cf_top10 = (
    cf_scores
    .sort_values(ascending=False)
    .head(10)
)

cf_recommendations = movies[
    movies["movieId"].isin(
        [
            itemidx_to_movieid[idx]
            for idx in cf_top10.index
        ]
    )
][["movieId", "title", "genres"]].copy()

# Preserve recommendation ranking
cf_recommendations["item_idx"] = (
    cf_recommendations["movieId"]
    .map(movieid_to_itemidx)
)

cf_recommendations["score"] = (
    cf_recommendations["item_idx"]
    .map(cf_top10)
)

cf_recommendations = (
    cf_recommendations
    .sort_values("score", ascending=False)
    .drop(columns="item_idx")
)

display(cf_recommendations)

,movieId,title,genres,score
1939,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller,18.773996
659,858,"Godfather, The (1972)",Crime|Drama,14.470933
3638,4993,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy,13.801385
2078,2762,"Sixth Sense, The (1999)",Drama|Horror|Mystery,13.426919
3141,4226,Memento (2000),Mystery|Thriller,12.491678
510,593,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,12.442969
4137,5952,"Lord of the Rings: The Two Towers, The (2002)",Adventure|Fantasy,12.111354
2996,4011,Snatch (2000),Comedy|Crime|Thriller,10.524831
2145,2858,American Beauty (1999),Drama|Romance,9.927730
3614,4963,Ocean's Eleven (2001),Crime|Thriller,9.712759


In [ ]:
# Step K3: Evaluate item-item CF

# Candidate pool for evaluation
cf_eval_scores = cf_scores.copy()

# Make sure held-out positive movies are allowed as candidates
cf_eval_scores = cf_eval_scores[
    ~cf_eval_scores.index.isin(
        personal_low_rated_ids
        .intersection(
            set(movieid_to_itemidx.keys())
        )
    )
]

cf_eval_top10 = (
    cf_eval_scores
    .sort_values(ascending=False)
    .head(10)
)

# Convert recommendations to item indices
cf_eval_items = list(cf_eval_top10.index)

# Calculate metrics
cf_hr10 = (
    1.0
    if len(set(cf_eval_items) & personal_test_pos) > 0
    else 0.0
)

cf_map10 = apk(
    personal_test_pos,
    cf_eval_items,
    k=10
)

cf_ndcg10 = ndcg_at_k(
    personal_test_pos,
    cf_eval_items,
    k=10
)

print(f"Item-item CF HR@10:   {cf_hr10:.4f}")
print(f"Item-item CF MAP@10:  {cf_map10:.4f}")
print(f"Item-item CF NDCG@10: {cf_ndcg10:.4f}")

print("\nHeld-out positive movies:")
display(
    movies[
        movies["movieId"].isin(
            [
                itemidx_to_movieid[idx]
                for idx in personal_test_pos
            ]
        )
    ][["movieId", "title", "genres"]]
)

Item-item CF HR@10:   1.0000
Item-item CF MAP@10:  0.1495
Item-item CF NDCG@10: 0.3254

Held-out positive movies:


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
126,153,Batman Forever (1995),Action|Adventure|Comedy|Crime
322,364,"Lion King, The (1994)",Adventure|Animation|Children|Drama|Musical|IMAX
510,593,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller
902,1200,Aliens (1986),Action|Adventure|Horror|Sci-Fi
1284,1704,Good Will Hunting (1997),Drama|Romance
3194,4306,Shrek (2001),Adventure|Animation|Children|Comedy|Fantasy|Ro...
3614,4963,Ocean's Eleven (2001),Crime|Thriller
3638,4993,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy
4066,5796,Casino Royale (1967),Action|Adventure|Comedy


In [34]:
# Show the top 10 item-item CF recommendations
cf_eval_display = movies[
    movies["movieId"].isin(
        [
            itemidx_to_movieid[idx]
            for idx in cf_eval_top10.index
        ]
    )
][["movieId", "title", "genres"]].copy()

cf_eval_display["item_idx"] = (
    cf_eval_display["movieId"]
    .map(movieid_to_itemidx)
)

cf_eval_display["score"] = (
    cf_eval_display["item_idx"]
    .map(cf_eval_top10)
)

cf_eval_display["is_test_positive"] = (
    cf_eval_display["item_idx"]
    .isin(personal_test_pos)
)

cf_eval_display = (
    cf_eval_display
    .sort_values("score", ascending=False)
    .drop(columns="item_idx")
)

display(cf_eval_display)

,movieId,title,genres,score,is_test_positive
1939,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller,18.773996,False
659,858,"Godfather, The (1972)",Crime|Drama,14.470933,False
3638,4993,"Lord of the Rings: The Fellowship of the Ring,...",Adventure|Fantasy,13.801385,True
2078,2762,"Sixth Sense, The (1999)",Drama|Horror|Mystery,13.426919,False
3141,4226,Memento (2000),Mystery|Thriller,12.491678,False
510,593,"Silence of the Lambs, The (1991)",Crime|Horror|Thriller,12.442969,True
4137,5952,"Lord of the Rings: The Two Towers, The (2002)",Adventure|Fantasy,12.111354,True
2996,4011,Snatch (2000),Comedy|Crime|Thriller,10.524831,False
2145,2858,American Beauty (1999),Drama|Romance,9.927730,False
3614,4963,Ocean's Eleven (2001),Crime|Thriller,9.712759,True


In [ ]:
# Step K4: Test neighbourhood size

from sklearn.neighbors import NearestNeighbors

k_values = [10, 25, 50, 100]

cf_results = []

for k in k_values:

    # Fit item-item model
    knn = NearestNeighbors(
        metric="cosine",
        algorithm="brute",
        n_neighbors=k + 1
    )

    knn.fit(ml_item_user)

    distances_k, indices_k = knn.kneighbors(
        ml_item_user
    )

    similarities_k = 1 - distances_k

    # Generate scores from personal training movies
    scores = defaultdict(float)

    for _, row in personal_profile.iterrows():

        source_item = row["movie_idx"]
        rating_weight = row["rating"] - 3

        for similarity, similar_item in zip(
            similarities_k[source_item][1:],
            indices_k[source_item][1:]
        ):
            scores[similar_item] += (
                similarity * rating_weight
            )

    scores = pd.Series(scores)

    # Exclude training positives
    scores = scores[
        ~scores.index.isin(personal_train_pos)
    ]

    # Exclude low-rated movies
    low_rated_item_ids = set(
        personal_ratings.loc[
            personal_ratings["rating"] < 4,
            "movieId"
        ].astype(str)
        .map(movieid_to_itemidx)
    )

    scores = scores[
        ~scores.index.isin(low_rated_item_ids)
    ]

    ranked = (
        scores
        .sort_values(ascending=False)
    )

    top10 = list(ranked.head(10).index)

    hr = (
        1.0
        if len(set(top10) & personal_test_pos) > 0
        else 0.0
    )

    map_score = apk(
        personal_test_pos,
        top10,
        k=10
    )

    ndcg = ndcg_at_k(
        personal_test_pos,
        top10,
        k=10
    )

    cf_results.append({
        "K": k,
        "HR@10": hr,
        "MAP@10": map_score,
        "NDCG@10": ndcg
    })

cf_results_df = pd.DataFrame(cf_results)

display(cf_results_df)

,K,HR@10,MAP@10,NDCG@10
0,10,1.0,0.176190,0.347629
1,25,1.0,0.133333,0.283233
2,50,1.0,0.149524,0.325429
3,100,1.0,0.230476,0.410572


In [ ]:
# Step K5: Test rating weighting

weighting_methods = {
    "linear": lambda r: r - 3,
    "centered": lambda r: r - 3.5,
    "squared": lambda r: (r - 3) ** 2
}

weight_results = []

# Reuse K=100 similarity model
k = 100

knn = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=k + 1
)

knn.fit(ml_item_user)

distances_100, indices_100 = knn.kneighbors(
    ml_item_user
)

similarities_100 = 1 - distances_100

low_rated_item_ids = set(
    personal_ratings.loc[
        personal_ratings["rating"] < 4,
        "movieId"
    ].astype(str)
    .map(movieid_to_itemidx)
)

for method_name, weight_function in weighting_methods.items():

    scores = defaultdict(float)

    for _, row in personal_profile.iterrows():

        source_item = row["movie_idx"]
        rating_weight = weight_function(row["rating"])

        for similarity, similar_item in zip(
            similarities_100[source_item][1:],
            indices_100[source_item][1:]
        ):
            scores[similar_item] += (
                similarity * rating_weight
            )

    scores = pd.Series(scores)

    scores = scores[
        ~scores.index.isin(personal_train_pos)
    ]

    scores = scores[
        ~scores.index.isin(low_rated_item_ids)
    ]

    ranked = scores.sort_values(ascending=False)
    top10 = list(ranked.head(10).index)

    hr = (
        1.0
        if len(set(top10) & personal_test_pos) > 0
        else 0.0
    )

    map_score = apk(
        personal_test_pos,
        top10,
        k=10
    )

    ndcg = ndcg_at_k(
        personal_test_pos,
        top10,
        k=10
    )

    weight_results.append({
        "Weighting": method_name,
        "HR@10": hr,
        "MAP@10": map_score,
        "NDCG@10": ndcg
    })

weight_results_df = pd.DataFrame(weight_results)

display(weight_results_df)

,Weighting,HR@10,MAP@10,NDCG@10
0,linear,1.0,0.230476,0.410572
1,centered,1.0,0.246032,0.422851
2,squared,1.0,0.262976,0.432773


In [ ]:
# Step K6: Build the final personal recommender

# Final item-item CF recommender
# Selected configuration:
# K = 100
# Cosine similarity
# Squared rating weighting

FINAL_K = 100

# Fit final item-item model
final_knn = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=FINAL_K + 1
)

final_knn.fit(ml_item_user)

final_distances, final_indices = final_knn.kneighbors(
    ml_item_user
)

final_similarities = 1 - final_distances

# Use all of your ratings to build your preference profile
final_profile = my_ratings.copy()

final_profile["movieId"] = (
    final_profile["movieId"].astype(str)
)

final_profile["movie_idx"] = movie_enc.transform(
    final_profile["movieId"]
)

# Accumulate recommendation scores
final_scores = defaultdict(float)

for _, row in final_profile.iterrows():

    source_item = row["movie_idx"]

    # Stronger ratings get substantially more influence
    rating_weight = (row["rating"] - 3) ** 2

    for similarity, similar_item in zip(
        final_similarities[source_item][1:],
        final_indices[source_item][1:]
    ):
        final_scores[similar_item] += (
            similarity * rating_weight
        )

# Convert to Series
final_scores = pd.Series(final_scores)

# Exclude every movie you have already rated
rated_item_indices = set(
    movie_enc.transform(
        my_ratings["movieId"].astype(str)
    )
)

final_scores = final_scores[
    ~final_scores.index.isin(rated_item_indices)
]

# Rank recommendations
final_ranked = (
    final_scores
    .sort_values(ascending=False)
)

# Top 20 recommendations
final_top20 = final_ranked.head(20)

# Create readable output
final_recommendations = movies[
    movies["movieId"].isin(
        [
            itemidx_to_movieid[idx]
            for idx in final_top20.index
        ]
    )
][["movieId", "title", "genres"]].copy()

final_recommendations["item_idx"] = (
    final_recommendations["movieId"]
    .map(movieid_to_itemidx)
)

final_recommendations["score"] = (
    final_recommendations["item_idx"]
    .map(final_top20)
)

final_recommendations = (
    final_recommendations
    .sort_values("score", ascending=False)
    .drop(columns="item_idx")
)

final_recommendations.insert(
    0,
    "rank",
    range(1, len(final_recommendations) + 1)
)

display(final_recommendations)

,rank,movieId,title,genres,score
1939,1,2571,"Matrix, The (1999)",Action|Sci-Fi|Thriller,43.567286
659,2,858,"Godfather, The (1972)",Crime|Drama,37.729345
2078,3,2762,"Sixth Sense, The (1999)",Drama|Horror|Mystery,36.205256
3141,4,4226,Memento (2000),Mystery|Thriller,33.252310
2996,5,4011,Snatch (2000),Comedy|Crime|Thriller,31.843432
969,6,1270,Back to the Future (1985),Adventure|Comedy|Sci-Fi,31.730726
2145,7,2858,American Beauty (1999),Drama|Romance,30.126818
3640,8,4995,"Beautiful Mind, A (2001)",Drama|Romance,29.117134
1298,9,1732,"Big Lebowski, The (1998)",Comedy|Crime,28.734460
3562,10,4878,Donnie Darko (2001),Drama|Mystery|Sci-Fi|Thriller,27.736994


In [ ]:
# Step K7: Build explanations for the final recommendations

rated_movie_titles = (
    movies.assign(movieId=lambda x: x["movieId"].astype(str))
    .set_index("movieId")["title"]
    .to_dict()
)

explanations = defaultdict(list)

final_candidate_set = set(final_top20.index)

for _, row in final_profile.iterrows():
    source_item = row["movie_idx"]
    source_movie_id = itemidx_to_movieid[source_item]
    source_rating = row["rating"]

    rating_weight = (source_rating - 3) ** 2

    for similarity, similar_item in zip(
        final_similarities[source_item][1:],
        final_indices[source_item][1:]
    ):
        if similar_item in final_candidate_set:
            contribution = similarity * rating_weight

            explanations[similar_item].append({
                "source_movie": rated_movie_titles.get(
                    source_movie_id,
                    source_movie_id
                ),
                "source_rating": source_rating,
                "similarity": similarity,
                "contribution": contribution
            })

In [ ]:
# Step K8: Display explanations for the top recommendations

for item_idx in final_top20.head(10).index:
    recommended_title = rated_movie_titles.get(
        itemidx_to_movieid[item_idx],
        itemidx_to_movieid[item_idx]
    )

    top_sources = sorted(
        explanations[item_idx],
        key=lambda x: x["contribution"],
        reverse=True
    )[:3]

    print(f"\n{recommended_title}")
    print("-" * len(recommended_title))

    for source in top_sources:
        print(
            f"Because you rated {source['source_movie']} "
            f"{source['source_rating']:.1f}/5 "
            f"(similarity: {source['similarity']:.3f})"
        )


Matrix, The (1999)
------------------
Because you rated Fight Club (1999) 5.0/5 (similarity: 0.568)
Because you rated Lord of the Rings: The Return of the King, The (2003) 5.0/5 (similarity: 0.523)
Because you rated Pulp Fiction (1994) 5.0/5 (similarity: 0.514)

Godfather, The (1972)
---------------------
Because you rated Pulp Fiction (1994) 5.0/5 (similarity: 0.546)
Because you rated Usual Suspects, The (1995) 5.0/5 (similarity: 0.512)
Because you rated Fight Club (1999) 5.0/5 (similarity: 0.507)

Sixth Sense, The (1999)
-----------------------
Because you rated Lord of the Rings: The Return of the King, The (2003) 5.0/5 (similarity: 0.461)
Because you rated Fight Club (1999) 5.0/5 (similarity: 0.456)
Because you rated Silence of the Lambs, The (1991) 5.0/5 (similarity: 0.445)

Memento (2000)
--------------
Because you rated Fight Club (1999) 5.0/5 (similarity: 0.513)
Because you rated Usual Suspects, The (1995) 5.0/5 (similarity: 0.450)
Because you rated Lord of the Rings: The Retu

In [ ]:
# Step K9: Turn it into an actual recommender

def recommend_movies(ratings_df, top_n=10, k=100):
    """
    Generate personalized movie recommendations using item-item
    collaborative filtering.
    """

    ratings = ratings_df.copy()
    ratings["movieId"] = ratings["movieId"].astype(str)

    ratings["movie_idx"] = movie_enc.transform(
        ratings["movieId"]
    )

    scores = defaultdict(float)

    explanations = defaultdict(list)

    for _, row in ratings.iterrows():
        source_item = row["movie_idx"]
        source_movie_id = itemidx_to_movieid[source_item]
        source_rating = row["rating"]

        rating_weight = (source_rating - 3) ** 2

        for similarity, similar_item in zip(
            final_similarities[source_item][1:k+1],
            final_indices[source_item][1:k+1]
        ):
            contribution = similarity * rating_weight

            scores[similar_item] += contribution

            explanations[similar_item].append({
                "source_movie": rated_movie_titles.get(
                    source_movie_id,
                    source_movie_id
                ),
                "source_rating": source_rating,
                "similarity": similarity,
                "contribution": contribution
            })

    scores = pd.Series(scores)

    rated_items = set(ratings["movie_idx"])

    scores = scores[
        ~scores.index.isin(rated_items)
    ]

    top_scores = scores.sort_values(
        ascending=False
    ).head(top_n)

    recommendations = movies[
        movies["movieId"].isin(
            [
                itemidx_to_movieid[idx]
                for idx in top_scores.index
            ]
        )
    ][["movieId", "title", "genres"]].copy()

    recommendations["item_idx"] = (
        recommendations["movieId"]
        .map(movieid_to_itemidx)
    )

    recommendations["score"] = (
        recommendations["item_idx"]
        .map(top_scores)
    )

    recommendations = (
        recommendations
        .sort_values("score", ascending=False)
        .drop(columns="item_idx")
    )

    recommendations.insert(
        0,
        "rank",
        range(1, len(recommendations) + 1)
    )

    return recommendations, explanations


recommendations, explanations = recommend_movies(
    my_ratings,
    top_n=10
)

display(recommendations)

In [ ]:
# Step K10: Add explanations to the recommendations


def display_recommendations(
    recommendations,
    explanations,
    top_sources=3
):
    """
    Display recommendations together with simple explanations.
    """

    for _, row in recommendations.iterrows():

        item_idx = movieid_to_itemidx[str(row["movieId"])]

        top_explanations = sorted(
            explanations[item_idx],
            key=lambda x: x["contribution"],
            reverse=True
        )[:top_sources]

        source_movies = [
            source["source_movie"]
            for source in top_explanations
        ]

        print(f"{int(row['rank'])}. {row['title']}")

        print(
            "   Recommended because you rated "
            + ", ".join(source_movies[:-1])
            + (
                f", and {source_movies[-1]}"
                if len(source_movies) > 1
                else source_movies[0]
            )
            + " highly."
        )

        print()


display_recommendations(
    recommendations,
    explanations
)

In [ ]:
# Step K11: Robustness Check with Positive-Only Rating Weighting

def evaluate_item_cf(rating_weight_function, k=100):
    scores = defaultdict(float)

    for _, row in personal_train.iterrows():
        source_item = row["movie_idx"]
        rating_weight = rating_weight_function(row["rating"])

        for similarity, similar_item in zip(
            final_similarities[source_item][1:k+1],
            final_indices[source_item][1:k+1]
        ):
            scores[similar_item] += similarity * rating_weight

    scores = pd.Series(scores)

    # Exclude movies already seen in training
    scores = scores[
        ~scores.index.isin(personal_train["movie_idx"])
    ]

    ranked_items = scores.sort_values(
        ascending=False
    ).head(10).index.tolist()

    actual = set(personal_test_pos)

    return {
        "HR@10": int(any(item in actual for item in ranked_items)),
        "MAP@10": apk(actual, ranked_items, k=10),
        "NDCG@10": ndcg_at_k(actual, ranked_items, k=10)
    }


positive_only_results = evaluate_item_cf(
    lambda rating: max(rating - 3, 0) ** 2,
    k=100
)

current_results = evaluate_item_cf(
    lambda rating: (rating - 3) ** 2,
    k=100
)

comparison = pd.DataFrame(
    [current_results, positive_only_results],
    index=[
        "Current: (rating - 3)^2",
        "Positive-only: max(rating - 3, 0)^2"
    ]
)

display(comparison)

,HR@10,MAP@10,NDCG@10
Current: (rating - 3)^2,1,0.262976,0.432773
"Positive-only: max(rating - 3, 0)^2",1,0.262976,0.432773


## Model Development

Several recommendation approaches were tested before selecting the final
model.

1. BPR collaborative filtering was first evaluated on MovieLens and
   outperformed a popularity baseline.

2. Personalized BPR was tested using my own ratings, but the limited number
   of personal interactions made it unsuitable for reliable
   personalization.

3. A genre-based content recommender was tested, but genre information was
   too coarse and produced weak personalization.

4. Item-item collaborative filtering was then evaluated. It uses the
   collective behaviour in MovieLens to learn relationships between movies,
   while my own ratings determine which of those relationships are most
   relevant to me.

This approach produced both strong holdout performance and recommendations
that were qualitatively aligned with my preferences, so it was selected as
the final recommender.

# Final Recommender

The final recommender uses item-item collaborative filtering to generate
personalized movie recommendations.

Movie similarities are learned from collective MovieLens user behaviour.
These similarities are then combined with my own movie ratings, with higher
ratings receiving greater weight.

The recommender excludes movies I have already rated and returns the highest
scoring unseen movies.

For each recommendation, the model also provides an explanation based on
movies I previously rated highly.

## Final Model

| Component | Choice |
|---|---|
| Data | MovieLens + personal ratings |
| Recommendation method | Item-item collaborative filtering |
| Similarity | Cosine similarity |
| Neighbours | 100 |
| Rating weighting | `(rating - 3)^2` |
| Candidate filtering | Exclude already-rated movies |
| Personal ratings | 199 |
| Evaluation | HR@10, MAP@10, NDCG@10 |

In [ ]:
## Final Recommendations

recommendations, explanations = recommend_movies(
    my_ratings,
    top_n=10
)

display_recommendations(
    recommendations,
    explanations
)

1. Matrix, The (1999)
   Recommended because you rated Fight Club (1999), Lord of the Rings: The Return of the King, The (2003), and Pulp Fiction (1994) highly.

2. Godfather, The (1972)
   Recommended because you rated Pulp Fiction (1994), Usual Suspects, The (1995), and Fight Club (1999) highly.

3. Sixth Sense, The (1999)
   Recommended because you rated Lord of the Rings: The Return of the King, The (2003), Fight Club (1999), and Silence of the Lambs, The (1991) highly.

4. Memento (2000)
   Recommended because you rated Fight Club (1999), Usual Suspects, The (1995), and Lord of the Rings: The Return of the King, The (2003) highly.

5. Snatch (2000)
   Recommended because you rated Fight Club (1999), Lord of the Rings: The Return of the King, The (2003), and Gladiator (2000) highly.

6. Back to the Future (1985)
   Recommended because you rated Departed, The (2006), Silence of the Lambs, The (1991), and Lord of the Rings: The Return of the King, The (2003) highly.

7. American Beau

## Personal Evaluation

Using a 80/20 holdout of my positively rated movies, the final
item-item collaborative filtering model achieved:

- HR@10: 1.000
- MAP@10: 0.263
- NDCG@10: 0.433

These results should be interpreted cautiously because the personal
evaluation is based on only 17 held-out positive ratings. The final
recommendations were also evaluated qualitatively based on whether they
matched my actual movie preferences.